# Learned Ensemble Reranker (LambdaMART)

**Why.** Hand-tuned score fusion (α=0.4 · clf + 0.6 · hybrid) lifted TREC22 full-corpus NDCG@10
from 0.458 to **0.554**. This generalizes that: a LambdaMART model over richer per-document
features, trained with a **listwise objective that optimizes NDCG directly** (the thing pointwise
cross-entropy never did). Cheap — LightGBM on CPU, minutes.

**Features per (topic, doc):** BM25 score+rank, dense cosine+rank, RRF score, clf-v4 P(relevant),
P(partial). All computed over the full-corpus hybrid candidate pool — the real serve distribution.

**Train:** TREC21+KZ candidate pools (labels from qrels, unjudged=0).  **Test:** TREC22 (held out).

**Baseline to beat:** hybrid fusion α=0.4 = 0.5540. This is also the ensemble slot where the
criterion signal / an LLM reranker later join as extra feature columns.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q lightgbm pytrec_eval rank-bm25 sentence-transformers datasets transformers tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATA_ROOT      = '/content/drive/MyDrive/ct_data23'
EVAL_ROOT      = f'{DATA_ROOT}/evaluation'
TREC_ROOT      = f'{EVAL_ROOT}/trec_data'
KZ_ROOT        = f'{EVAL_ROOT}/kz_data'
FULLTEXT_CORPUS = f'{DATA_ROOT}/doc_texts_fulltext.txt'
DENSE_EMB_FILE = f'{DATA_ROOT}/doc_embeddings_retriever-v2_fulltext.npy'
BM25_CACHE     = f'{DATA_ROOT}/bm25_fulltext.pkl'
CLF_CHECKPOINT = 'semaj83/ctmatch-clf-v4'
RETRIEVER_MODEL = 'semaj83/ctmatch-retriever-v2'
MODEL_OUT      = f'{DATA_ROOT}/ensemble_ltr_v1.txt'   # LightGBM booster

TRAIN_SETS     = ['trec21', 'kz']
TEST_SETS      = ['trec22']
CAND_K         = 1000    # candidate pool depth per topic (union of bm25 + dense top-K, via RRF)
os.environ['CTMATCH_DATA_ROOT'] = DATA_ROOT
print('config set')

In [ ]:
import pickle, numpy as np, torch, torch.nn.functional as F
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from ctmatch.evaluation.eval_utils import load_eval_datasets

# corpus (full-text) aligned to index2docid order
_idx = load_dataset('semaj83/ctmatch_ir', data_files='index2docid.txt', split='train')
corpus_ids = [r['text'].strip() for r in _idx]
with open(FULLTEXT_CORPUS) as f:
    corpus_txt = [l.rstrip('\n') for l in f]
docid2text = {d: t for d, t in zip(corpus_ids, corpus_txt)}

# BM25 index (cached), dense embeddings (cached), retriever + clf
with open(BM25_CACHE, 'rb') as f:
    bm25 = pickle.load(f)
doc_emb = np.load(DENSE_EMB_FILE).astype(np.float32)
doc_emb /= (np.linalg.norm(doc_emb, axis=1, keepdims=True) + 1e-9)
q_encoder = SentenceTransformer(RETRIEVER_MODEL)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
clf_tok = AutoTokenizer.from_pretrained(CLF_CHECKPOINT)
clf = AutoModelForSequenceClassification.from_pretrained(CLF_CHECKPOINT).to(device).eval()
name2idx = {v.lower(): int(k) for k, v in clf.config.id2label.items()}
IDX_REL = name2idx.get('relevant'); IDX_PARTIAL = next((i for n, i in name2idx.items() if 'partial' in n), None)

all_sets = load_eval_datasets(TREC_ROOT, KZ_ROOT)
print('sets:', list(all_sets.keys()), '| relevant idx', IDX_REL, '| partial idx', IDX_PARTIAL)

## Feature extraction

For each topic: full BM25 + dense scores over the corpus, take the union of their top-`CAND_K`
as the candidate pool (RRF for the fusion feature), then clf-score only that pool. One feature
row per candidate; label from qrels (unjudged = 0). ~20-30 min on GPU across all sets.

In [ ]:
from tqdm.auto import tqdm

FEATURES = ['bm25', 'bm25_rank', 'dense', 'dense_rank', 'rrf', 'clf_rel', 'clf_partial']

def clf_scores(topic_text, idxs, batch=64):
    rel, par = [], []
    for i in range(0, len(idxs), batch):
        texts = [corpus_txt[j] for j in idxs[i:i+batch]]
        enc = clf_tok([topic_text]*len(texts), texts, padding=True, truncation=True,
                      max_length=512, return_tensors='pt').to(device)
        with torch.no_grad():
            p = F.softmax(clf(**enc).logits, -1).cpu().numpy()
        rel.extend(p[:, IDX_REL]); par.extend(p[:, IDX_PARTIAL])
    return rel, par

def build_rows(sets):
    X, y, groups, meta = [], [], [], []   # meta: (set, tid, doc_id)
    for name in sets:
        ds = all_sets[name]
        for tid, text in tqdm(ds['topic2text'].items(), desc=f'features {name}'):
            if tid not in ds['rel_dict']:
                continue
            qv = q_encoder.encode(text, normalize_embeddings=True).astype(np.float32)
            bm = np.array(bm25.index.get_scores(bm25._tokenize(text)))     # full corpus
            dn = doc_emb @ qv
            bm_top = np.argpartition(-bm, CAND_K)[:CAND_K]
            dn_top = np.argpartition(-dn, CAND_K)[:CAND_K]
            cand = list(set(bm_top.tolist()) | set(dn_top.tolist()))
            # ranks within each retriever (dense/bm25) for RRF + rank features
            bm_rank = {j: r for r, j in enumerate(sorted(cand, key=lambda j: -bm[j]))}
            dn_rank = {j: r for r, j in enumerate(sorted(cand, key=lambda j: -dn[j]))}
            rel, par = clf_scores(text, cand)
            rel_dict = ds['rel_dict'][tid]
            n = 0
            for k, j in enumerate(cand):
                rrf = 1.0/(60+bm_rank[j]+1) + 1.0/(60+dn_rank[j]+1)
                X.append([bm[j], bm_rank[j], float(dn[j]), dn_rank[j], rrf, rel[k], par[k]])
                y.append(int(rel_dict.get(corpus_ids[j], 0)))
                meta.append((name, tid, corpus_ids[j])); n += 1
            groups.append(n)
    return np.array(X, dtype=np.float32), np.array(y), groups, meta

Xtr, ytr, gtr, mtr = build_rows(TRAIN_SETS)
Xte, yte, gte, mte = build_rows(TEST_SETS)
print(f'\ntrain rows {len(ytr):,} ({len(gtr)} topics) | test rows {len(yte):,} ({len(gte)} topics)')
print('train label dist:', {int(v): int((ytr==v).sum()) for v in [0,1,2]})

In [ ]:
import lightgbm as lgb

dtrain = lgb.Dataset(Xtr, label=ytr, group=gtr, feature_name=FEATURES)
params = {
    'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [10],
    'learning_rate': 0.05, 'num_leaves': 31, 'min_data_in_leaf': 50,
    'lambda_l2': 1.0, 'max_position': 20, 'verbose': -1,
}
booster = lgb.train(params, dtrain, num_boost_round=300)
booster.save_model(MODEL_OUT)
print('saved ->', MODEL_OUT)
print('\nfeature importance (gain):')
for f, imp in sorted(zip(FEATURES, booster.feature_importance('gain')), key=lambda x: -x[1]):
    print(f'  {f:12s} {imp:12.1f}')

In [ ]:
import pytrec_eval

pred = booster.predict(Xte)
run = {}
for score, (name, tid, doc) in zip(pred, mte):
    run.setdefault(tid, {})[doc] = float(score)
qrel = {tid: {d: int(r) for d, r in all_sets[TEST_SETS[0]]['rel_dict'][tid].items()} for tid in run}
ev = pytrec_eval.RelevanceEvaluator(qrel, {'ndcg_cut.10', 'recip_rank', 'P.10'})
res = ev.evaluate(run)
nd = np.mean([v['ndcg_cut_10'] for v in res.values()])
mr = np.mean([v['recip_rank'] for v in res.values()])
print(f'LambdaMART ensemble — TREC22 NDCG@10={nd:.4f}  MRR={mr:.4f}')
print('\nBaselines: hybrid fusion alpha=0.4 = 0.5540 | full-text BM25->clf = 0.458 | TrialGPT ~0.73')